In [1]:
!pip install transformers datasets torch torchvision


  Using cached numpy-2.2.6-cp310-cp310-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached urllib

In [2]:
from transformers import AutoImageProcessor, SwinForImageClassification
from datasets import load_dataset
import torch

/Users/dhaneshkumarkapadia/Desktop/swinTransformer/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
model_name = "microsoft/swin-tiny-patch4-window7-224"
image_processor = AutoImageProcessor.from_pretrained(model_name)
model = SwinForImageClassification.from_pretrained(model_name)

Loading weights: 100%|██████████| 233/233 [00:00<00:00, 1998.18it/s, Materializing param=swin.layernorm.weight]                                                     


In [26]:
dataset = load_dataset("cifar10", split="test[:1000]")

In [27]:
images = [item["img"] for item in dataset]
labels = [item["label"] for item in dataset]

In [28]:
from transformers import AutoImageProcessor, AutoModelForImageClassification, TrainingArguments, Trainer

image_processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForImageClassification.from_pretrained(
model_name, num_labels=10, ignore_mismatched_sizes=True
)

Loading weights: 100%|██████████| 233/233 [00:00<00:00, 1990.53it/s, Materializing param=swin.layernorm.weight]                                                     
SwinForImageClassification LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([10])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([10, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [18]:
inputs = image_processor(images, return_tensors="pt").to(model.device)

In [29]:
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

In [30]:
predicted_labels = logits.argmax(dim=-1).cpu().numpy()

In [31]:
num_classes = len(model.config.id2label)
if num_classes != len(set(labels)):
    print("Warning: Model label space does not match CIFAR-10 labels. Mapping may be required.")
    class_mapping = {i: i % 10 for i in range(num_classes)}
    predicted_labels = [class_mapping[label] for label in predicted_labels]

In [32]:
class_names = [
    "airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"
]
predicted_class_names = [class_names[label] for label in predicted_labels]
true_class_names = [class_names[label] for label in labels]

#print(predicted_class_names)
#print(true_class_names)

In [23]:
for i, (true_label, predicted_label) in enumerate(zip(true_class_names, predicted_class_names)):
    print(
        f"Image {i + 1}: True Label = {true_label}, Predicted Label = {predicted_label}")

Image 1: True Label = cat, Predicted Label = dog
Image 2: True Label = ship, Predicted Label = automobile
Image 3: True Label = ship, Predicted Label = ship
Image 4: True Label = airplane, Predicted Label = bird
Image 5: True Label = frog, Predicted Label = bird
Image 6: True Label = frog, Predicted Label = ship
Image 7: True Label = automobile, Predicted Label = dog
Image 8: True Label = frog, Predicted Label = automobile
Image 9: True Label = cat, Predicted Label = dog
Image 10: True Label = automobile, Predicted Label = dog
Image 11: True Label = airplane, Predicted Label = frog
Image 12: True Label = truck, Predicted Label = dog
Image 13: True Label = dog, Predicted Label = horse
Image 14: True Label = horse, Predicted Label = dog
Image 15: True Label = truck, Predicted Label = dog
Image 16: True Label = ship, Predicted Label = airplane
Image 17: True Label = dog, Predicted Label = dog
Image 18: True Label = horse, Predicted Label = truck
Image 19: True Label = ship, Predicted Labe

In [34]:
from sklearn.metrics import confusion_matrix, classification_report


cm = confusion_matrix(true_class_names, predicted_class_names)
print(cm)

print(classification_report(true_class_names, predicted_class_names))

[[15  3 46  7 11  3  4  4  7  3]
 [ 5  1 69  3 10  0  0  1  0  0]
 [13  9 49  8  5  0  0  9  1  6]
 [16  5 44  6 17  0  1  1  8  5]
 [29  5 47  0  2  2  0  4  0  1]
 [ 8  0 30 22  5  6  2  6  7  0]
 [17  1 57  4  9 12  0  5  5  2]
 [ 9  0 73  3 11  1  0  1  3  1]
 [28  2 57  7  8  0  1  2  1  0]
 [ 3  0 65  2 26  1  0  3  9  0]]
              precision    recall  f1-score   support

    airplane       0.10      0.15      0.12       103
  automobile       0.04      0.01      0.02        89
        bird       0.09      0.49      0.15       100
         cat       0.10      0.06      0.07       103
        deer       0.02      0.02      0.02        90
         dog       0.24      0.07      0.11        86
        frog       0.00      0.00      0.00       112
       horse       0.03      0.01      0.01       102
        ship       0.02      0.01      0.01       106
       truck       0.00      0.00      0.00       109

    accuracy                           0.08      1000
   macro avg       